<a href="https://colab.research.google.com/github/hyang0129/NGAFIDDATASET/blob/main/NGAFID_DATASET_TF_EXAMPLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INSTALL PREREQ

In [1]:
!git clone https://github.com/ccDT2022/NGAFIDDATASET.git

!(cd NGAFIDDATASET ; git checkout main; git reset --hard HEAD; git pull)
!(cd NGAFIDDATASET ; pip install -r requirements.txt -q)



Cloning into 'NGAFIDDATASET'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 47 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 496.50 KiB | 6.80 MiB/s, done.
Resolving deltas: 100% (18/18), done.
Already on 'main'
Your branch is up to date with 'origin/main'.
HEAD is now at 32262a3 Add files via upload
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 26.5

# IMPORT DEPENDENCIES

In [2]:
# 1. 导入必要的系统库
import sys
import importlib

# 2. 打补丁：将 importlib 伪装成 Python 3.12 中已经被移除的 imp 模块
sys.modules['imp'] = importlib

# 3. 补充原代码逻辑：将下载好的数据集项目路径添加到系统环境变量中
sys.path.append('/content/NGAFIDDATASET')

# 4. 此时系统里已经有了一个“假”的 imp，可以安全地加载 autoreload 扩展了
%load_ext autoreload

In [3]:
from tqdm.autonotebook import tqdm


/tmp/ipykernel_15512/527081995.py:1: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [4]:
%autoreload
from ngafiddataset.dataset.dataset import NGAFID_Dataset_Manager
from ngafiddataset.dataset.utils import to_dict_of_list
from ngafiddataset.utils import connect_to_tpu
import pandas as pd


strategy = connect_to_tpu()

/content/NGAFIDDATASET/ngafiddataset/dataset/dataset.py:55: SyntaxWarning: invalid escape sequence '\o'
  logger.info('Downloading and extracting Parquet Files to %s\one_parq.  Please open them using dask dataframes' % destination)


No TPU Available
REPLICAS:  1


# DEFINE MODEL FN

In [5]:
import tensorflow as tf

tfk = tf.keras
tfkl = tf.keras.layers

class Classifier_INCEPTION:
    def __init__(
        self,
        input_shape,
        nb_classes,
        build=True,
        batch_size=64,
        nb_filters=32,
        use_residual=True,
        use_bottleneck=True,
        depth=6,
        kernel_size=41,
        nb_epochs=1500,
        two_output = False,
        mode = None
    ):

        self.nb_filters = nb_filters
        self.use_residual = use_residual
        self.use_bottleneck = use_bottleneck
        self.depth = depth
        self.kernel_size = kernel_size - 1
        self.callbacks = None
        self.batch_size = batch_size
        self.bottleneck_size = 32
        self.nb_epochs = nb_epochs
        self.two_output = two_output
        self.mode = mode

        if build is True:
            self.model = self.build_model(input_shape, nb_classes)

    def _inception_module(self, input_tensor, stride=1, activation="linear"):

        if self.use_bottleneck and int(input_tensor.shape[-1]) > 1:
            input_inception = tf.keras.layers.Conv1D(
                filters=self.bottleneck_size, kernel_size=1, padding="same", activation=activation, use_bias=False
            )(input_tensor)
        else:
            input_inception = input_tensor

        # kernel_size_s = [3, 5, 8, 11, 17]
        kernel_size_s = [self.kernel_size // (2 ** i) for i in range(3)]

        conv_list = []

        for i in range(len(kernel_size_s)):
            conv_list.append(
                tf.keras.layers.Conv1D(
                    filters=self.nb_filters,
                    kernel_size=kernel_size_s[i],
                    strides=stride,
                    padding="same",
                    activation=activation,
                    use_bias=False,
                )(input_inception)
            )

        max_pool_1 = tf.keras.layers.MaxPool1D(pool_size=3, strides=stride, padding="same")(input_tensor)

        conv_6 = tf.keras.layers.Conv1D(
            filters=self.nb_filters, kernel_size=1, padding="same", activation=activation, use_bias=False
        )(max_pool_1)

        conv_list.append(conv_6)

        x = tf.keras.layers.Concatenate(axis=2)(conv_list)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation(activation="relu")(x)
        return x

    def _shortcut_layer(self, input_tensor, out_tensor):
        shortcut_y = tf.keras.layers.Conv1D(
            filters=int(out_tensor.shape[-1]), kernel_size=1, padding="same", use_bias=False
        )(input_tensor)
        shortcut_y = tf.keras.layers.BatchNormalization()(shortcut_y)

        x = tf.keras.layers.Add()([shortcut_y, out_tensor])
        x = tf.keras.layers.Activation("relu")(x)
        return x

    def build_model(self, input_shape, nb_classes):
        input_layer = tf.keras.layers.Input(input_shape, name = 'data')

        x = input_layer
        input_res = input_layer

        for d in range(self.depth):

            x = self._inception_module(x)

            if self.use_residual and d % 3 == 2:
                x = self._shortcut_layer(input_res, x)
                input_res = x

        gap_layer = tf.keras.layers.GlobalAveragePooling1D()(x)


        outputs = []

        outputs.append(tf.keras.layers.Dense(nb_classes, activation="softmax", name='target_class')(gap_layer))

        if self.two_output:
            outputs.append(tf.keras.layers.Dense(1, activation="sigmoid", name='before_after')(gap_layer))


        if self.mode == 'before_after':
            outputs = []
            outputs.append(tf.keras.layers.Dense(1, activation="sigmoid", name='before_after')(gap_layer))

        model = tf.keras.models.Model(inputs=input_layer, outputs=outputs)

        return model

    def get_kernel_model(self):
        '''
        Get a model whose output is just the global average pooling layer.

        Returns:

        '''

        return tf.keras.Model(self.model.layers[0].input, self.model.layers[-2].output)




In [6]:
def point_wise_feed_forward_network(d_model, dff):
  return tf.keras.Sequential([
      tf.keras.layers.Dense(dff, activation='relu'),  # (batch_size, seq_len, dff)
      tf.keras.layers.Dense(d_model)  # (batch_size, seq_len, d_model)
  ])

def scaled_dot_product_attention(q, k, v, mask):
  """Calculate the attention weights.
  q, k, v must have matching leading dimensions.
  k, v must have matching penultimate dimension, i.e.: seq_len_k = seq_len_v.
  The mask has different shapes depending on its type(padding or look ahead)
  but it must be broadcastable for addition.

  Args:
    q: query shape == (..., seq_len_q, depth)
    k: key shape == (..., seq_len_k, depth)
    v: value shape == (..., seq_len_v, depth_v)
    mask: Float tensor with shape broadcastable
          to (..., seq_len_q, seq_len_k). Defaults to None.

  Returns:
    output, attention_weights
  """

  matmul_qk = tf.matmul(q, k, transpose_b=True)  # (..., seq_len_q, seq_len_k)

  # scale matmul_qk
  dk = tf.cast(tf.shape(k)[-1], tf.float32)
  scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

  # add the mask to the scaled tensor.
  if mask is not None:
    scaled_attention_logits += (mask * -1e9)

  # softmax is normalized on the last axis (seq_len_k) so that the scores
  # add up to 1.
  attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)  # (..., seq_len_q, seq_len_k)

  output = tf.matmul(attention_weights, v)  # (..., seq_len_q, depth_v)

  return output, attention_weights

class MultiHeadAttention(tf.keras.layers.Layer):
  def __init__(self, d_model, num_heads):
    super(MultiHeadAttention, self).__init__()
    self.num_heads = num_heads
    self.d_model = d_model

    assert d_model % self.num_heads == 0

    self.depth = d_model // self.num_heads

    self.wq = tf.keras.layers.Dense(d_model)
    self.wk = tf.keras.layers.Dense(d_model)
    self.wv = tf.keras.layers.Dense(d_model)

    self.dense = tf.keras.layers.Dense(d_model)

  def split_heads(self, x, batch_size):
    """Split the last dimension into (num_heads, depth).
    Transpose the result such that the shape is (batch_size, num_heads, seq_len, depth)
    """
    x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
    return tf.transpose(x, perm=[0, 2, 1, 3])

  def call(self, v, k, q, mask):
    batch_size = tf.shape(q)[0]

    q = self.wq(q)  # (batch_size, seq_len, d_model)
    k = self.wk(k)  # (batch_size, seq_len, d_model)
    v = self.wv(v)  # (batch_size, seq_len, d_model)

    q = self.split_heads(q, batch_size)  # (batch_size, num_heads, seq_len_q, depth)
    k = self.split_heads(k, batch_size)  # (batch_size, num_heads, seq_len_k, depth)
    v = self.split_heads(v, batch_size)  # (batch_size, num_heads, seq_len_v, depth)

    # scaled_attention.shape == (batch_size, num_heads, seq_len_q, depth)
    # attention_weights.shape == (batch_size, num_heads, seq_len_q, seq_len_k)
    scaled_attention, attention_weights = scaled_dot_product_attention(
        q, k, v, mask)

    scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])  # (batch_size, seq_len_q, num_heads, depth)

    concat_attention = tf.reshape(scaled_attention,
                                  (batch_size, -1, self.d_model))  # (batch_size, seq_len_q, d_model)

    output = self.dense(concat_attention)  # (batch_size, seq_len_q, d_model)

    return output, attention_weights

class EncoderLayer(tf.keras.layers.Layer):
  def __init__(self, d_model, num_heads, dff, rate=0.1):
    super(EncoderLayer, self).__init__()

    self.mha = MultiHeadAttention(d_model, num_heads)
    self.ffn = point_wise_feed_forward_network(d_model, dff)

    self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
    self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    self.dropout1 = tf.keras.layers.Dropout(rate)
    self.dropout2 = tf.keras.layers.Dropout(rate)

  #def call(self, x, training, mask = None):
  def call(self, x, training=None, mask=None):

    attn_output, self.attention_values = self.mha(x, x, x, mask)  # (batch_size, input_seq_len, d_model)
    attn_output = self.dropout1(attn_output, training=training)
    out1 = self.layernorm1(x + attn_output)  # (batch_size, input_seq_len, d_model)

    ffn_output = self.ffn(out1)  # (batch_size, input_seq_len, d_model)
    ffn_output = self.dropout2(ffn_output, training=training)
    out2 = self.layernorm2(out1 + ffn_output)  # (batch_size, input_seq_len, d_model)

    return out2

def get_angles(pos, i, d_model):
    angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(d_model))
    return pos * angle_rates

def positional_encoding(position, d_model):
    angle_rads = get_angles(np.arange(position)[:, np.newaxis],
                            np.arange(d_model)[np.newaxis, :],
                            d_model)

    # apply sin to even indices in the array; 2i
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

    # apply cos to odd indices in the array; 2i+1
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    pos_encoding = angle_rads[np.newaxis, ...]

    return tf.cast(pos_encoding, dtype=tf.float32)

class PositionalEncoding(tf.keras.layers.Layer):

    def __init__(self, maximum_position_encoding = 1000):
        super(PositionalEncoding, self).__init__()

        self.pos_encoding = positional_encoding(maximum_position_encoding,
                                        512)

    def call(self, x):

        seq_len = tf.shape(x)[1]
        x += self.pos_encoding[:, :seq_len, :]

        return x

def get_mhsa_model(nbclass = 2, SHAPE = (4096, 23), mode = 'before_after'):
    # I realize that I did not include a positional embedding, but its too late now. Maybe a future paper could address this?
    d = 512
    ff = 1024
    model =  tfk.Sequential([tf.keras.Input(shape = SHAPE),
                                            tfkl.Conv1D(128, 7, strides = 1, padding='same', activation='relu'),
                                            # tfkl.BatchNormalization(),
                                            tfkl.Conv1D(128, 7, strides = 2, padding='same', activation='relu'),
                                            # tfkl.BatchNormalization(),
                                            tfkl.Conv1D(256, 7, strides = 1, padding='same', activation='relu'),
                                            # tfkl.BatchNormalization(),
                                            tfkl.Conv1D(256, 7, strides = 2, padding='same', activation='relu'),
                                            # tfkl.BatchNormalization(),
                                            # tfkl.Conv1D(512, 7, strides = 1, padding='same', activation='relu'),
                                            tfkl.Conv1D(512, 7, strides = 2, padding='same', activation='relu'),
                                            # tfkl.Conv1D(768, 7, strides = 1, padding='same', activation='relu'),
                                            # tfkl.BatchNormalization(),
                                            EncoderLayer(d_model=d, num_heads=8, dff=ff),
                                            EncoderLayer(d_model=d, num_heads=8, dff=ff),
                                            EncoderLayer(d_model=d, num_heads=8, dff=ff),
                                            EncoderLayer(d_model=d, num_heads=8, dff=ff),

                                            tf.keras.layers.GlobalAveragePooling1D(),
                                            # tfkl.Dense(nbclass, activation='softmax'),
    ])


    input = tf.keras.Input(shape = SHAPE, name = 'data')

    x = model(input)

    output = []
    if mode == 'before_after':
        output =  tfkl.Dense(1, activation='sigmoid', name = 'before_after')(x)
    elif mode == 'both':
        output.append(tf.keras.layers.Dense(nbclass, activation="softmax", name='target_class')(x))
        output.append(tfkl.Dense(1, activation='sigmoid', name = 'before_after')(x))
    elif mode == 'classes':
        output.append(tf.keras.layers.Dense(nbclass, activation="softmax", name='target_class')(x))

    fun_model = tf.keras.Model(input, output)

    return fun_model

def get_mhsa_model_pe():
    # I realize that I did not include a positional embedding, but its too late now. Maybe a future paper could address this?

    model =  tfk.Sequential([tf.keras.Input(shape = SHAPE),
                                            tfkl.Conv1D(128, 7, strides = 1, padding='same', activation='relu'),
                                            tfkl.BatchNormalization(),
                                            tfkl.Conv1D(128, 7, strides = 2, padding='same', activation='relu'),
                                            tfkl.BatchNormalization(),
                                            tfkl.Conv1D(256, 3, strides = 1, padding='same', activation='relu'),
                                            tfkl.BatchNormalization(),
                                            tfkl.Conv1D(256, 7, strides = 2, padding='same', activation='relu'),
                                            tfkl.BatchNormalization(),
                                            tfkl.Conv1D(512, 7, strides = 2, padding='same', activation='relu'),
                                            tfkl.BatchNormalization(),
                                            # tfkl.Conv1D(512, 7, strides = 2, padding='same', activation='relu'),
                                            # tfkl.Conv1D(512, 7, strides = 2, padding='same', activation='relu'),
                                            PositionalEncoding(),
                                            EncoderLayer(d_model=512, num_heads=8, dff=512),
                                            EncoderLayer(d_model=512, num_heads=8, dff=512),
                                            EncoderLayer(d_model=512, num_heads=8, dff=512),
                                            EncoderLayer(d_model=512, num_heads=8, dff=512),
                                            # tfkl.Lambda(lambda x : x[:, 0, :]),
                                            tf.keras.layers.GlobalAveragePooling1D(),
                                            tfkl.Dense(1, activation='sigmoid'),
    ])

    return model


model = get_mhsa_model(mode = 'both')
model.summary()



Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ data (InputLayer)   │ (None, 4096, 23)  │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_4        │ (None, 512)       │ 10,153,344 │ data[0][0]        │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ target_class        │ (None, 2)         │      1,026 │ sequential_4[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ before_after        │ (None, 1)         │        513 │ sequential_4[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,154,883 (38.74 MB)

 Trainable params: 10,154,883 (38.74 MB)

 Non-trainable params: 0 (0.00 B)

# SETUP DATA

In [7]:
# 1. 删除刚才下载失败的“伪装网页”文件
!rm -f /content/2days.tar.gz
!rm -rf /content/2days

# 2. 使用正确的直链接口，强制下载真实的 2days 数据集
!gdown "https://drive.google.com/uc?id=1OlvdT5xk9-mvFnlhfXYydxRnEUtVe-yH" -O /content/2days.tar.gz

# 3. 将下载好的压缩包解压到当前目录
!tar -xzf /content/2days.tar.gz -C /content/

print("✅ 2days 数据集下载并解压完成！")

# 4. 继续运行原本的数据加载逻辑（关键：加上 extract=False 防止它再次尝试解压）
from ngafiddataset.dataset.dataset import NGAFID_Dataset_Manager
import pandas as pd

dm = NGAFID_Dataset_Manager('2days', extract=False)
df = pd.read_csv('/content/2days/flight_header.csv')

# 构建数据字典 (注意 TF 版本的例子这里用的是 numpy=False)
dm.data_dict = dm.construct_data_dictionary(numpy=False)


Downloading...
From (original): https://drive.google.com/uc?id=1OlvdT5xk9-mvFnlhfXYydxRnEUtVe-yH
From (redirected): https://drive.google.com/uc?id=1OlvdT5xk9-mvFnlhfXYydxRnEUtVe-yH&confirm=t&uuid=eafbcbb7-96a4-44ae-9267-d30f51eb31f4
To: /content/2days.tar.gz
100% 1.13G/1.13G [00:13<00:00, 85.2MB/s]
✅ 2days 数据集下载并解压完成！


  0%|          | 0/11446 [00:00<?, ?it/s]

In [8]:
number_classes = len(dm.flight_header_df['class'].unique())
number_classes

number_hierarchies = len(dm.flight_header_df['hclass'].unique())
number_hierarchies


5

# DEFINE TASK

In [9]:
mode = 'before_after' #开启维护前vs维护后的二分类任务
# mode = 'hierarchy_basic'
#mode = 'both'  #同时输出分类结果（target_class）和二分类结果（before_after）。
# mode = 'classes' #多分类任务
#Baseline 复现使用 mode = 'before_after'，完成维护事件二分类检测
#改进方面使用mode = 'both'（多任务学习），通过引入故障类型的标签作为辅助任务，模型能够学习到更细致的故障特征，从而反过来提升二分类（维护前/后）的判断准确度。

# DETERMINE MODEL STRUCTURE BASED ON OUTPUTS
two_output = False
if mode == 'before_after':
    nb_classes = 1
elif mode == 'hierarchy_basic':
    nb_classes = number_hierarchies + 1
    two_output = True
elif mode == 'both':
    two_output = True
    nb_classes = number_classes+1
else:
    nb_classes = number_classes+1

# TRAIN MODEL

In [10]:

class Saver(tf.keras.callbacks.Callback):

    def __init__(self, model_name):
        super().__init__()
        self.model_name = model_name
        # self.bucket = bucket
        # self.fs = gcsfs.GCSFileSystem(project='tpu-44747', token = 'gckey.json')
        self.start = 5
        self.best = 0

    def on_epoch_end(self, epoch, logs=None):
        if epoch > self.start:
            pass

        metric = 'val_loss'

        try:
            if logs.get(metric) >= 0.85 and logs.get(metric) > self.best:
                print('saving good model')
                lpath = self.model_name + '.h5'
                self.model.save(lpath, include_optimizer=False)
                self.best = logs.get(metric)
        except Exception as E:
            print('encountered some error in model saving process')
            print(E)
            pass


In [11]:
import numpy as np
import pandas as pd
from tqdm.autonotebook import tqdm
model_name = 'ITIME_BOTH'
# model_name = 'CONVMHSA_BOTH'
save_path = ''
all_fold_accuracies = [] # 用于存储每折的最佳准确率

flight_res = []


for fold in tqdm(range(5)):

    save_filename = save_path + '%s_%i' % (model_name, fold)

    #减小 batch_size，避免发生OOM
    #train_ds = dm.get_tf_dataset(fold = fold, training = True, shuffle = 1000, repeat = True, mode = mode, batch_size = 128)
    train_ds = dm.get_tf_dataset(fold = fold, training = True, shuffle = 1000, repeat = True, mode = mode, batch_size = 64)
    #test_ds = dm.get_tf_dataset(fold = fold, training  = False, shuffle = False, repeat = False, mode = mode, batch_size = 128)
    test_ds = dm.get_tf_dataset(fold = fold, training  = False, shuffle = False, repeat = False, mode = mode, batch_size = 64)



    with strategy.scope():
        #在训练中使用 InceptionTime 模型，默认为InceptionTime
        model = Classifier_INCEPTION(input_shape = (4096, 23), nb_classes=nb_classes, two_output = two_output ).model
        lr = 1e-4
        #在训练中使用 ConvMHSA 模型
        # model = get_mhsa_model(nbclass=nb_classes, mode = mode)
        # lr = 3e-5

        if two_output:

            model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=lr),
                        metrics =
                                    {'target_class' : [tf.keras.metrics.SparseCategoricalAccuracy(name = 'multi_acc')],
                                    'before_after' : [tf.keras.metrics.BinaryAccuracy(name = 'single_acc')],
                                    }

                        ,
                        loss = {'target_class' : tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False, name = 'multi_loss'),
                                'before_after' : tf.keras.losses.BinaryCrossentropy(from_logits=False, name = 'single_loss')}
            )


        elif mode == 'before_after':

            model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=lr),
                        metrics = [
                                    tf.keras.metrics.BinaryAccuracy()
                        ],
                        loss = tf.keras.losses.BinaryCrossentropy(from_logits=False))


        elif mode == 'classes':

            model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=lr),
                        metrics = [
                                    tf.keras.metrics.SparseCategoricalAccuracy(name = 'multi_acc')
                        ],
                        loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False))

        else:
            raise


        callbacks = [
                tf.keras.callbacks.ModelCheckpoint(
                    filepath="best.weights.h5",
                    save_best_only=True,  # Only save a model if `val_loss` has improved.
                    monitor="val_loss",
                    verbose=1,
                    save_weights_only=True
                ),
                # 碍于电脑性能，加入下面这行自动早停代码，patience=15，表示如果15轮没进步就提前结束这200轮的训练
                tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

            ]


        history = model.fit(
            train_ds,
            steps_per_epoch = 100,
            #在epochs=85发生卡顿，原来为200，现改为30.因为5轮为150
            epochs = 30,
            validation_data = test_ds,
            callbacks = callbacks,
            verbose = True,

        )

        history = pd.DataFrame(history.history)
        history['epoch'] = history.index
        history['model'] = model_name
        history.to_csv(save_filename)

        #model.load_weights('best.h5')
        model.load_weights('best.weights.h5')
        # --- 新增：评估并记录本折准确率 ---
        eval_results = model.evaluate(test_ds, verbose=0)

        # 提取准确率：根据 compile 时的定义，通常索引 1 是准确率
        # 如果是 multi-output (both)，single_acc 通常在最后
        fold_acc = eval_results[-1] if two_output or mode == 'both' else eval_results[1]
        all_fold_accuracies.append(fold_acc)
        print(f"Fold {fold} Best Val Accuracy: {fold_acc:.4f}")

        '''
        flights_before_acc = {}

        for day in range(5):


            indices = list(dm.flight_header_df[ (dm.flight_header_df['number_flights_before'] == day) & (dm.flight_header_df.before_after == 1) & (dm.flight_header_df.fold == fold) & (dm.flight_header_df.label == 'intake gasket leak/damage')].index)
            print(len(indices))
            ds = tf.data.Dataset.from_tensor_slices(to_dict_of_list([example for example in dm.data_dict if example['id'] in indices]))

            ds = dm.get_tf_dataset(ds = ds , fold = 0, training  = False, shuffle = False, repeat = False, mode = mode, batch_size = 2)

            res = model.evaluate(ds)

            flights_before_acc[day] = res[-1]

        flight_res.append(flights_before_acc)
        '''
        # --- 新增：最终报告打印 ---
if all_fold_accuracies:
    mean_acc = np.mean(all_fold_accuracies)
    std_acc = np.std(all_fold_accuracies)

    print("\n" + "="*40)
    print("      最终交叉验证报告 (CV REPORT)      ")
    print("="*40)
    for i, acc in enumerate(all_fold_accuracies):
        print(f"Fold {i}: 准确率 = {acc:.4f}")
    print("-" * 40)
    print(f"平均准确率 (Mean): {mean_acc:.4f}")
    print(f"标准差 (Std Dev):  {std_acc:.4f}")
    print(f"综合表现: {mean_acc:.4f} ± {std_acc:.4f}")
    print("="*40)


  0%|          | 0/5 [00:00<?, ?it/s]

<unknown>:55: SyntaxWarning: invalid escape sequence '\o'
<unknown>:55: SyntaxWarning: invalid escape sequence '\o'


Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (64, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step - binary_accuracy: 0.5014 - loss: 0.7136
Epoch 1: val_loss improved from None to 0.70661, saving model to best.weights.h5

Epoch 1: finished saving model to best.weights.h5
100/100 ━━━━━━━━━━━━━━━━━━━━ 75s 534ms/step - binary_accuracy: 0.4903 - loss: 0.7004 - val_binary_accuracy: 0.4893 - val_loss: 0.7066
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 530ms/step - binary_accuracy: 0.4889 - loss: 0.6788
Epoch 2: val_loss improved from 0.70661 to 0.70300, saving model to best.weights.h5

Epoch 2: finished saving model to best.weights.h5
100/100 ━━━━━━━━━━━━━━━━━━━━ 56s 558ms/step - binary_accuracy: 0.4841 - loss: 0.6705 - val_binary_accuracy: 0.4893 - val_loss: 0.7030
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 519ms/step - binary_accuracy: 0.5032 - loss: 0.6850
Epoch 3: val_loss did not improve from 0.70300
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 546ms/step - binary_accuracy: 0.4967 - loss: 0.6736 - val_binary_accuracy: 0.4893 - val_loss: 0.7588
Epoch 4/30

...

100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 531ms/step - binary_accuracy: 0.4921 - loss: 0.7259
Epoch 1: val_loss improved from None to 0.69138, saving model to best.weights.h5

Epoch 1: finished saving model to best.weights.h5
100/100 ━━━━━━━━━━━━━━━━━━━━ 70s 574ms/step - binary_accuracy: 0.4881 - loss: 0.7054 - val_binary_accuracy: 0.4920 - val_loss: 0.6914
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 517ms/step - binary_accuracy: 0.4912 - loss: 0.6823
Epoch 2: val_loss did not improve from 0.69138
100/100 ━━━━━━━━━━━━━━━━━━━━ 54s 544ms/step - binary_accuracy: 0.4866 - loss: 0.6796 - val_binary_accuracy: 0.4920 - val_loss: 0.6948
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 523ms/step - binary_accuracy: 0.4939 - loss: 0.6728
Epoch 3: val_loss improved from 0.69138 to 0.68921, saving model to best.weights.h5

Epoch 3: finished saving model to best.weights.h5
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 550ms/step - binary_accuracy: 0.4919 - loss: 0.6664 - val_binary_accuracy: 0.4920 - val_loss: 0.6892
Epoch 4/30

/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (64, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 531ms/step - binary_accuracy: 0.4973 - loss: 0.7121
Epoch 1: val_loss improved from None to 0.71555, saving model to best.weights.h5

Epoch 1: finished saving model to best.weights.h5
100/100 ━━━━━━━━━━━━━━━━━━━━ 70s 575ms/step - binary_accuracy: 0.4934 - loss: 0.6989 - val_binary_accuracy: 0.4795 - val_loss: 0.7156
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 518ms/step - binary_accuracy: 0.4857 - loss: 0.6842
Epoch 2: val_loss improved from 0.71555 to 0.71001, saving model to best.weights.h5

Epoch 2: finished saving model to best.weights.h5
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 546ms/step - binary_accuracy: 0.4905 - loss: 0.6769 - val_binary_accuracy: 0.4795 - val_loss: 0.7100
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 522ms/step - binary_accuracy: 0.4903 - loss: 0.6650
Epoch 3: val_loss did not improve from 0.71001
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 548ms/step - binary_accuracy: 0.4939 - loss: 0.6674 - val_binary_accuracy: 0.4795 - val_loss: 0.8561
Epoch 4/30

/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (64, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 530ms/step - binary_accuracy: 0.4889 - loss: 0.7023
Epoch 1: val_loss improved from None to 0.69196, saving model to best.weights.h5

Epoch 1: finished saving model to best.weights.h5
100/100 ━━━━━━━━━━━━━━━━━━━━ 70s 574ms/step - binary_accuracy: 0.4888 - loss: 0.6972 - val_binary_accuracy: 0.4732 - val_loss: 0.6920
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 519ms/step - binary_accuracy: 0.5019 - loss: 0.6838
Epoch 2: val_loss did not improve from 0.69196
100/100 ━━━━━━━━━━━━━━━━━━━━ 54s 545ms/step - binary_accuracy: 0.4945 - loss: 0.6762 - val_binary_accuracy: 0.4732 - val_loss: 0.6920
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 522ms/step - binary_accuracy: 0.4965 - loss: 0.6782
Epoch 3: val_loss did not improve from 0.69196
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 548ms/step - binary_accuracy: 0.4945 - loss: 0.6736 - val_binary_accuracy: 0.4732 - val_loss: 0.7291
Epoch 4/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 522ms/step - binary_accuracy: 0.4939 - loss: 0.6607
Epo

In [12]:
#flight_res

In [13]:
#print(flight_res)